# Aggregate Metadata

In [ ]:
environment = 'icad'

%load_ext autoreload
%autoreload 1

%aimport highlighting_precision
%aimport gradcam_APs
%aimport vanilla_vgg
%aimport dataHelper
%aimport dataHandling

%matplotlib inline

In [ ]:
# imports
import numpy as np
import pandas as pd
from tqdm import tqdm
import ast
import matplotlib.pyplot as plt
import torch
import os
from pathlib import Path

In [ ]:
from dataHandling import find_dicoms

dicom_path = Path('/redacted/path/to/dicoms')
external_validation_path = Path('/redacted/path/to/external/validation')
ge_path = dicom_path / 'ge' / 'image'

dcm_paths = find_dicoms(ge_path)
dcm_paths[:5]

In [ ]:
from dataHandling import truthfile_for
# selected dicom with associated mass annotations for an example
truthfile = truthfile_for(dcm_paths[32])
!ls -ltr $truthfile
!cat $truthfile

In [ ]:
!dirname $truthfile

In [ ]:
# Test Parsing an entire record with masses
from dataHandling import parse_truth, icad_patient_from_path, icad_side_view_from_path

with open(truthfile, 'r') as f:
    side, position = icad_side_view_from_path(truthfile)
    records = parse_truth(f.read(), icad_patient_from_path(truthfile), 'ge', side, position)

assert len(records) == 1
icad_patient_from_path
r = records[0]
assert r['mass_anno_link_exists'] == True, r['mass_anno_link']
records[0]

In [ ]:
# Test Parsing individual annotation
from dataHandling import __parse_path_annotation

annotation_path = Path('/redacted/path/to/some.ann.txt')

assert annotation_path.exists()

__parse_path_annotation(annotation_path)

In [ ]:
from dataHandling import icad_vendor_from_path
assert icad_patient_from_path(truthfile) == 'redacted', icad_patient_from_path(truthfile)
assert icad_patient_from_path(annotation_path) == 'redacted', icad_patient_from_path(annotation_path)

assert icad_vendor_from_path(truthfile) == 'ge', truthfile
assert icad_vendor_from_path(annotation_path) == 'ge', annotation_path

In [ ]:
# Test parsing
for i, dcm_path in enumerate(dcm_paths):
    truthfile = truthfile_for(dcm_path)
    try:
        with open(truthfile, 'r') as f:
            xml = f.read()
            records = parse_truth(xml, icad_patient_from_path(truthfile), icad_vendor_from_path(truthfile), 'L', 'MLO')
    except Exception as e:
        if type(e) == FileNotFoundError:
            print(dcm_path)
        else:
            print(i)
            print(xml)
            raise


## Construct Raw Validation Dataframe

In [ ]:
pd.set_option('display.max_rows', 15)

In [ ]:
from dataHandling import pandas_for_icad

truth_df = pandas_for_icad(dicom_path)

print(truth_df.shape)
truth_df

### Data Quality Checks

#### Annotation File - XML Consistency

##### Ensure All Annotation Files are the Same
Result: All annotation files are the same, no matter where they are referenced.

In [ ]:
truth_df['mass_anno_link_exists'].value_counts(dropna=False)

In [ ]:
# there are 3 different places we can have annotation files
truth_df['mass_core_anno_link_exists'].value_counts(dropna=False)

In [ ]:
truth_df['mass_spiculation_anno_link_exists'].value_counts(dropna=False)

In [ ]:
# And, if it has a link, then it has points, and vice versa.
truth_df['mass_path_len_nonzero'] = truth_df['mass_path_len'] > 0
truth_df.groupby(['mass_anno_link_exists', 'mass_path_len_nonzero']).count()[['birads_truthed']]

In [ ]:
# Check that the links always point to the same files
link_cols = ['mass_anno_link', 'mass_core_anno_link', 'mass_spiculation_anno_link']

def isnan(maybe_str):
    if type(maybe_str) == float:
        return True
    else:
        return False

def anno_equal_or_na(row):
    anno = row['mass_anno_link']
    core = row['mass_core_anno_link']
    spic = row['mass_spiculation_anno_link']

    if not isnan(anno):
        if isnan(core):
            return isnan(spic) or anno == spic
        else:
            return core == anno and (core == spic or isnan(spic))
    else:
        if isnan(core):
            return True
        else:
            return core == spic or isnan(spic)

__match = truth_df.apply(anno_equal_or_na, axis=1).value_counts(dropna=False)

# All True means they're all the same, so we can use any one
assert False not in __match.keys(), __match
__match


##### Ensure Annotation Files Match the Points in XML
Result: they're all the same except round-off error, so we can use any source of the path annotation

In [ ]:
# Are the lengths the same in annotations and embedded points?
truth_df['mass_anno_path_len'] = truth_df['mass_anno_path_size()'].astype(np.float).fillna(0)
truth_df['mass_path_len'] = truth_df['mass_path_len'].astype(np.float).fillna(0)
path_len_match = (truth_df['mass_anno_path_len'] ==\
    truth_df['mass_path_len'].astype(np.float))

__match = path_len_match.value_counts()
assert False not in __match, __match
__match

In [ ]:
def compare_paths(path_a, path_b):
    if type(path_a) == float:
        return 0 if type(path_b) == float else np.inf
        
    if len(path_a) != len(path_b):
        return np.inf

    delta = 0.0
    for a, b in zip(path_a, path_b):
        for i in range(2):
            delta = delta + np.abs(np.round(a[i]) - np.round(b[i]))
    
    return delta

def __compare_base(row):
    return compare_paths(row['mass_anno_path_points'], row['mass_path_points'])

__match = truth_df.apply(__compare_base, axis=1)
__match[__match > 0].describe()
# max difference of 8 pixels after rounding is very close

In [ ]:
# Can use mass_anno_path_nchains to find things with ROIs because it's a superset of those with annotations 
cols = ['mass_anno_path_nchains', 'mass_spiculation_anno_path_nchains', 'mass_core_anno_path_nchains']
truth_df[cols + ['birads_truthed']].fillna(0).groupby(cols).count()

In [ ]:
def compare_core(row):
    return compare_paths(row['mass_core_anno_path_points'], row['mass_core_path_points'])

__match = truth_df.apply(compare_core, axis=1)
__match[__match > 0].describe()

In [ ]:
def compare_spic(row):
    return compare_paths(row['mass_spiculation_anno_path_points'], row['mass_spiculation_path_points'])

__match = truth_df.apply(compare_spic, axis=1)
__match[__match > 0].describe()

##### Other Path Checks

In [ ]:
# Only one path per mass
__match = truth_df['mass_anno_path_nchains'].value_counts(dropna=False)
assert False not in __match, __match
__match

In [ ]:
truth_df['mass_type[0]'] = truth_df['mass_type'].apply(lambda x: np.nan if type(x) == float else x[0])
truth_df.groupby(['mass_anno_link_exists', 'mass_spiculation_anno_link_exists', 'mass_anno_link_exists', 'mass_type[0]', 'mass_histo']).count()

In [ ]:
# For ROI Paths,
# 6000 (not spiculated) is a strict subset of core
# 6001 (spiculated) has both spiculation and core, but is a subset
# That means we can ignore the path_nchains and just use the mass_type label
cols = ['mass_anno_path_nchains', 'mass_spiculation_anno_path_nchains', 'mass_core_anno_path_nchains', 'mass_type[0]', 'mass_histo']
truth_df[truth_df['mass_type[0]'].isin(['6000', '6001'])][cols + ['birads_truthed']].fillna(0).groupby(cols).count()

## Pre-Clean Statistics

In [ ]:
truth_df['mass_type'].value_counts(dropna=False)

In [ ]:
truth_df['mass_birads_desc'].value_counts(dropna=False)

In [ ]:
truth_df['mass_birads_score'].value_counts(dropna=False)

In [ ]:
truth_df['mass_histo'].value_counts(dropna=False)

In [ ]:
truth_df['birads'].value_counts(dropna=False)

In [ ]:
truth_df['masses_count'].value_counts()

In [ ]:
# Can just use the set `manufacturer` column
truth_df.groupby(['ii_manufacturer', 'manufacturer']).count()

In [ ]:
test_data_path = Path('/redacted/path/to/test/data')
# Uncomment to regenerate
# truth_df.to_csv(test_data_path / 'truth_df.txt')

# Create Clean Dataframe

In [ ]:
keep_cols = ['patient_id', 'side', 'view', 'mass_lesion_uid', 'mass_uid', 'truthfile', 'dcm_file', 'manufacturer', 'acquistion_type', 'has_anantomical', 'glandular', 'birads', 'has_clusters', 'has_mixed', 'has_fps', 'ii_modality', 'ii_height', 'ii_width', 'masses_count', 'mass_detectable', 'mass_missed', 'mass_anno_link', 'mass_anno_path_points', 'mass_anno_path_nchains', 'mass_path_len', 'mass_histo', 'mass_birads_score']

In [ ]:
truth_df['malignancy_label'] = truth_df['mass_histo'].apply(lambda x: np.nan if type(x) == float else x == '3')
keep_cols.append('malignancy_label')
truth_df['malignancy_label'].value_counts(dropna=False)

In [ ]:
# convert mass type
margin_map = {
    '6000': 'not_spiculated',
    '6001': 'spiculated',
    '6002': 'arch_distortion',
    '6003': 'asym',
    '-2': np.nan
}

truth_df['mass_margin'] = truth_df['mass_type'].apply(lambda x: np.nan if type(x) == float else margin_map[x[0]])
keep_cols.append('mass_margin')
truth_df['mass_margin'].value_counts(dropna=False)

In [ ]:
truth_df['dcm_id'] = truth_df['dcm_file'].astype(str).str.split('/',expand=True)[11].str.split('.', expand=True)[0]
keep_cols.append('dcm_id')

truth_df['dcm_id'].head(3)

In [ ]:
cleaning_df = truth_df[keep_cols]
cleaning_df

### Select Relevant Data

In [ ]:
def stats(df):
    margin = df['mass_margin'].value_counts()
    pathology = df['malignancy_label'].value_counts()
    acquisition = df['acquistion_type'].value_counts()
    return pd.DataFrame([{'patients': df['patient_id'].nunique(),
        'masses': df['mass_uid'].nunique(),
        'images': df['dcm_file'].nunique(),
        'spiculated': margin['spiculated'] if 'spiculated' in margin else 0,
        'not_spiculated': margin['not_spiculated'] if 'not_spiculated' in margin else 0,
        'malignant': pathology[True] if True in pathology else 0,
        'benign': pathology[False] if False in pathology else 0,
        'screen': acquisition['1'],
        'diagnostic': acquisition['2'] if '2' in acquisition else 0
        }])[['patients', 'images', 'masses', 'malignant', 'benign', 'spiculated', 'not_spiculated', 'screen', 'diagnostic']]
cleaning_df.pipe(stats)

In [ ]:
scr_df = cleaning_df[(cleaning_df['acquistion_type'] == '1') & cleaning_df['view'].isin(['MLO', 'CC'])]
scr_df.pipe(stats)

In [ ]:
scr_2d_df = scr_df[scr_df['ii_modality'] == 'MG']
scr_2d_df.pipe(stats)

In [ ]:
scr_2d_mass_df = scr_2d_df[scr_2d_df['mass_path_len'] > 0]
scr_2d_mass_df.pipe(stats)

In [ ]:
scr_2d_mass_mal_df = scr_2d_mass_df[~scr_2d_mass_df['malignancy_label'].isna()]
scr_2d_mass_mal_df.pipe(stats)

In [ ]:
scr_2d_mass_mal_margin_df = scr_2d_mass_mal_df[scr_2d_mass_mal_df['mass_margin'].isin(['not_spiculated', 'spiculated'])]
scr_2d_mass_mal_margin_df.pipe(stats)

In [ ]:
clean_df = roi_margin_masses_df

clean_df.pipe(stats).pipe(display)
clean_df.groupby(['mass_margin', 'malignancy_label']).count()[['patient_id']]

In [ ]:
from dataHandling import path_to_bounding_box

def row_to_path_bounding_box(row):
    return path_to_bounding_box(row['ii_height'], row['ii_width'], row['mass_anno_path_points'])

clean_df['path_bounding_box'] = clean_df.apply(row_to_path_bounding_box, axis=1)
clean_df['path_bounding_box']

In [ ]:
# Uncomment to regenerate
# clean_df.to_csv(test_data_path / 'clean_df.csv')

# Copy Images

In [ ]:
# Sample Bounding Boxes
import pydicom
import cv2
from matplotlib.patches import Rectangle

for idx, row in clean_df.sample(5).iterrows():
    
    print(row['patient_id'])
    ds = pydicom.dcmread(str(row['dcm_file']))
    plt.imshow(ds.pixel_array, cmap='gray')
    for p in row['mass_anno_path_points']:
        plt.plot(p.x, p.y, 'r.')

    (x1, y1), (x2, y2) = row['path_bounding_box']
    
    bb = Rectangle((x1, y1), x2-x1, y2-y1, fill=False, ec='green')

    plt.gca().add_patch(bb)
    plt.show()

In [ ]:
from dataHandling import make_icad_iaiabl_dataset_folder

extraction_df = make_icad_iaiabl_dataset_folder(clean_df, test_data_path)
extraction_df

In [ ]:
__failed = extraction_df['failed'].value_counts(dropna=False)
assert True not in __failed, __failed
__failed

In [ ]:
%matplotlib inline
for plt_path in extraction_df.sample(5)['plt_path']:
    image = cv2.imread(str(plt_path))
    plt.imshow(image)
    plt.show()
    plt.close()

In [ ]:
%matplotlib inline
for plt_path in extraction_df.sample(5)['numpy_path']:
    roi = np.load(str(plt_path), allow_pickle=True)
    fig, axs = plt.subplots(1,2)
    axs[0].imshow(roi[0], cmap='gray')
    axs[1].imshow(roi[1], cmap='gray')
    plt.show()
    plt.close(fig)

# Subset Selection
Note that `clean_df` represents all the data that meet the criteria to be able to run IAIA-BL. There is still some subselection for quality that may be applied (for instance, excluding images acquired from diagnostics).

In [ ]:
image_df = clean_df.merge(extraction_df, on='mass_uid')
image_df.columns

# END

Go to `ICADEvaluation.ipynb` to see the results from running the models over this dataset.